In [1]:
!nvidia-smi

Fri Apr 24 22:20:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!git clone https://github.com/sanjana231100/hr-llm-alignment.git
%cd hr-llm-alignment

Cloning into 'hr-llm-alignment'...
remote: Enumerating objects: 46, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 46 (delta 11), reused 41 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (46/46), 35.50 KiB | 4.44 MiB/s, done.
Resolving deltas: 100% (11/11), done.
/kaggle/working/hr-llm-alignment


In [3]:
!pip install -q transformers peft trl datasets accelerate bitsandbytes wandb pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 28.1 MB/s eta 0:00:00


In [4]:
import os
import sys
import gc
import torch
import wandb
import accelerate
from datasets import load_dataset
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForSequenceClassification
from peft import LoraConfig, TaskType, prepare_model_for_kbit_training, get_peft_model
from trl import RewardTrainer, RewardConfig
import trl
print("TRL version:", trl.__version__)
print("imports OK")

TRL version: 1.2.0
imports OK


In [5]:
import os
os.environ["WANDB_API_KEY"] = "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"
os.environ["ACCELERATE_MIXED_PRECISION"] = "no"
import wandb
wandb.login(key=os.environ["WANDB_API_KEY"])
print("W&B logged in")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sanju-231100 (sanju-231100-santa-clara-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B logged in


In [6]:
print("Loading preference pairs...")
raw_dataset = load_dataset("Anthropic/hh-rlhf", split="train")
reward_dataset = raw_dataset.select(range(2000))
print(f"Reward dataset size: {len(reward_dataset)}")

Loading preference pairs...


README.md: 0.00B [00:00, ?B/s]

harmless-base/train.jsonl.gz:   0%|          | 0.00/13.2M [00:00<?, ?B/s]

helpful-base/train.jsonl.gz:   0%|          | 0.00/16.2M [00:00<?, ?B/s]

helpful-online/train.jsonl.gz:   0%|          | 0.00/20.1M [00:00<?, ?B/s]

...

helpful-online/test.jsonl.gz:   0%|          | 0.00/1.05M [00:00<?, ?B/s]

helpful-rejection-sampled/test.jsonl.gz:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/160800 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/8552 [00:00<?, ? examples/s]

Reward dataset size: 2000


In [7]:
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.1")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "right"
print("Tokenizer ready")

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Tokenizer ready


In [8]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
)

print("Loading reward model...")
reward_model = AutoModelForSequenceClassification.from_pretrained(
    "mistralai/Mistral-7B-v0.1",
    num_labels=1,
    quantization_config=bnb_config,
    device_map="auto",
)
reward_model.config.pad_token_id = tokenizer.pad_token_id
reward_model = prepare_model_for_kbit_training(reward_model)
reward_model = get_peft_model(reward_model, peft_config)

# patch score layer dtype
original_forward = reward_model.base_model.model.score.forward
def patched_forward(x, *args, **kwargs):
    x = x.to(torch.bfloat16)
    return original_forward(x, *args, **kwargs)
reward_model.base_model.model.score.forward = patched_forward

reward_model.print_trainable_parameters()

reward_config = RewardConfig(
    output_dir="/kaggle/working/outputs/reward",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-5,
    warmup_steps=10,
    logging_steps=10,
    save_steps=100,
    report_to="wandb",
    fp16=False,
    bf16=False,
    max_grad_norm=1.0,
    max_length=512,
    remove_unused_columns=False,
    run_name="reward-model-mistral-7b-hr",
)

trainer = RewardTrainer(
    model=reward_model,
    args=reward_config,
    train_dataset=reward_dataset,
    processing_class=tokenizer,
)

print("Starting reward model training...")
trainer.train()

print("Saving reward model...")
trainer.save_model("/kaggle/working/outputs/reward")
tokenizer.save_pretrained("/kaggle/working/outputs/reward")
print("Done.")

Loading reward model...


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

MistralForSequenceClassification LOAD REPORT from: mistralai/Mistral-7B-v0.1
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Filtering train >512 tokens:   0%|          | 0/2000 [00:00<?, ? examples/s]

Starting reward model training...


wandb: setting up run 5pmrf8kb
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/hr-llm-alignment/wandb/run-20260424_222218-5pmrf8kb
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run reward-model-mistral-7b-hr
wandb: ⭐️ View project at https://wandb.ai/sanju-231100-santa-clara-university/huggingface
wandb: 🚀 View run at https://wandb.ai/sanju-231100-santa-clara-university/huggingface/runs/5pmrf8kb


Step,Training Loss
10,1.852884
20,2.309044
30,2.297333
40,1.932596
50,2.253928
60,2.151063
70,1.949395
80,2.086864
90,1.675478
100,2.209202


Saving reward model...
Done.


In [9]:
import shutil
shutil.copytree(
    "/kaggle/working/outputs/reward",
    "/kaggle/working/reward-mistral-7b-hr",
    dirs_exist_ok=True
)
print("Reward model files:")
for f in os.listdir("/kaggle/working/reward-mistral-7b-hr"):
    print(f)

Reward model files:
adapter_config.json
training_args.bin
checkpoint-200
README.md
adapter_model.safetensors
tokenizer.json
checkpoint-100
checkpoint-243
tokenizer_config.json
